# 02 · Data Preprocessing

**Round 1 demo:** show the actual checks and transformations applied before feature engineering. The rules are applied trip by trip, preserve timestamp order, and leave source files untouched.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
DATA=ROOT/"data"/"Student folder"
print("Project:",ROOT,"| dataset found:",DATA.exists())

Project: /Users/yashas/Documents/Codex/2026-11-18/referenced-chatgpt-conversation-this-is-an-2/ML_hackathon | dataset found: True


## Step 1 — Parse each headerless file with an explicit schema

In [2]:
import numpy as np, pandas as pd
from pathsnap.data import read_trip,clean_track
trip=read_trip(DATA/"train"/"00000001","train")
raw_track=np.loadtxt(DATA/"train"/"00000001"/"00000001.track",dtype=float,ndmin=2)
print({"track_columns":["longitude","latitude","timestamp_s"],"node_columns":["longitude","latitude"],"arc_columns":["from_node_id","to_node_id"],"route_column":"arc_id","raw_track_shape":raw_track.shape,"cleaned_track_shape":trip.track.shape})
display(pd.DataFrame(raw_track[:6],columns=["longitude","latitude","timestamp_s"]))

{'track_columns': ['longitude', 'latitude', 'timestamp_s'], 'node_columns': ['longitude', 'latitude'], 'arc_columns': ['from_node_id', 'to_node_id'], 'route_column': 'arc_id', 'raw_track_shape': (1315, 3), 'cleaned_track_shape': (1315, 3)}


,longitude,latitude,timestamp_s
0,8.595203,49.844053,1.0
1,8.595294,49.844441,2.0
2,8.595352,49.844883,3.0
3,8.595390,49.845379,4.0
4,8.595587,49.845750,5.0
5,8.595555,49.846167,6.0


## Step 2 — Validate and clean GPS rows

Rules: reject non-finite values; enforce longitude in [-180, 180] and latitude in [-90, 90]; discard duplicate/backward timestamps without sorting. These checks are important even though the supplied trips currently appear clean.

In [3]:
cleaned=clean_track(raw_track)
finite=np.isfinite(raw_track).all(axis=1)
valid_bounds=(np.abs(raw_track[:,0])<=180)&(np.abs(raw_track[:,1])<=90)
print({"nonfinite_rows":int((~finite).sum()),"out_of_range_coordinate_rows":int((~valid_bounds).sum()),"nonincreasing_timestamp_steps":int(np.sum(np.diff(cleaned[:,2])<=0)),"rows_removed":len(raw_track)-len(cleaned),"rows_retained":len(cleaned)})
display(pd.DataFrame(cleaned[:6],columns=["longitude","latitude","timestamp_s"]))

{'nonfinite_rows': 0, 'out_of_range_coordinate_rows': 0, 'nonincreasing_timestamp_steps': 0, 'rows_removed': 0, 'rows_retained': 1315}


,longitude,latitude,timestamp_s
0,8.595203,49.844053,1.0
1,8.595294,49.844441,2.0
2,8.595352,49.844883,3.0
3,8.595390,49.845379,4.0
4,8.595587,49.845750,5.0
5,8.595555,49.846167,6.0


## Step 3 — Preserve gaps; calculate movement using elapsed time

We do not interpolate or compress a gap into a fake one-second interval. For consecutive observations, speed is displacement divided by the measured time difference. A long gap is flagged for later route decoding.

In [4]:
dt=np.diff(cleaned[:,2]); dlon=np.diff(cleaned[:,0]); dlat=np.diff(cleaned[:,1])
lat0=float(np.median(cleaned[:,1])); dx=dlon*111320*np.cos(np.deg2rad(lat0)); dy=dlat*110540
distance_m=np.hypot(dx,dy); speed_mps=distance_m/np.maximum(dt,1e-9)
steps=pd.DataFrame({"dt_s":dt,"displacement_m":distance_m,"speed_mps":speed_mps,"gap_over_5s":dt>5})
print("timestamp interval summary (s):"); display(steps.dt_s.describe(percentiles=[.5,.9,.95,.99]).to_frame())
print("derived speed summary (m/s):"); display(steps.speed_mps.describe(percentiles=[.5,.9,.95,.99]).to_frame())
print("long gaps retained:",int(steps.gap_over_5s.sum()))

timestamp interval summary (s):


,dt_s
count,1314.000000
mean,1.032725
std,0.446355
min,1.000000
50%,1.000000
90%,1.000000
95%,1.000000
99%,2.000000
max,16.000000


derived speed summary (m/s):


,speed_mps
count,1314.000000
mean,37.852294
std,15.617348
min,1.230945
50%,41.232472
90%,55.910770
95%,58.456231
99%,63.645950
max,71.996016


long gaps retained: 1


## Step 4 — Check road-network references and topology

In [5]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
def graph_summary(t):
  n=len(t.nodes); a=t.arcs
  valid=(a[:,0]>=0)&(a[:,1]>=0)&(a[:,0]<n)&(a[:,1]<n)
  av=a[valid].astype(int); degree=np.bincount(av.ravel(),minlength=n) if len(av) else np.zeros(n,dtype=int)
  g=coo_matrix((np.ones(len(av)),(av[:,0],av[:,1])),shape=(n,n)).tocsr()
  ncomp,labels=connected_components(g,directed=False,return_labels=True)
  route_ok=(t.route>=0)&(t.route<len(a))
  endpoint_ok=np.zeros(len(t.route),dtype=bool)
  ids=t.route[route_ok].astype(int); endpoint_ok[np.flatnonzero(route_ok)]=valid[ids]
  sizes=np.bincount(labels)
  return {"nodes":n,"arcs":len(a),"invalid_arc_rows":int((~valid).sum()),"isolated_nodes":int((degree==0).sum()),"weak_components":int(ncomp),"largest_component_fraction":float(sizes.max()/n),"route_ids_out_of_range":int((~route_ok).sum()),"route_arcs_with_invalid_endpoints":int((~endpoint_ok).sum())}
for path,part in [(DATA/"train"/"00000001","train"),(DATA/"student_test"/"00000000","student_test")]:
  print(part,graph_summary(read_trip(path,part)))

train {'nodes': 88809, 'arcs': 165790, 'invalid_arc_rows': 0, 'isolated_nodes': 0, 'weak_components': 175, 'largest_component_fraction': 0.9779864653357205, 'route_ids_out_of_range': 0, 'route_arcs_with_invalid_endpoints': 0}
student_test {'nodes': 8542, 'arcs': 18196, 'invalid_arc_rows': 0, 'isolated_nodes': 0, 'weak_components': 100, 'largest_component_fraction': 0.8740341840318426, 'route_ids_out_of_range': 0, 'route_arcs_with_invalid_endpoints': 0}


The component count is a weak-connectivity check (direction ignored); arc direction is retained later. Invalid arcs are excluded from geometric candidates. The supplied trip structures can include disconnected or isolated network areas, so downstream route assumptions need to be measured rather than presumed.

## Step 5 — Convert geographic coordinates to local metres

Longitude/latitude are angular coordinates, so we project around each trip’s median location for short urban-scale distances: `x = (lon − lon0) × 111,320 × cos(lat0)` and `y = (lat − lat0) × 110,540`. This local approximation supports candidate distances and segment projection over a trip-sized area. The map itself keeps longitude/latitude.

In [6]:
lon0=float(np.median(trip.track[:,0])); lat0=float(np.median(trip.track[:,1]))
def local_xy(lon,lat): return ((lon-lon0)*111320*np.cos(np.deg2rad(lat0)),(lat-lat0)*110540)
print("projection origin:",(lon0,lat0),"first GPS in local metres:",local_xy(*trip.track[0,:2]))

projection origin: (8.603564, 50.128292) first GPS in local metres: (np.float64(-596.674356618643), np.float64(-31419.779059999943))


### Preprocessing summary

This archive produced **zero** invalid-coordinate rows and **zero** non-increasing timestamp steps, so cleaning removed no rows. This is a measured outcome, not a reason to omit the cleaning code. Timestamp gaps were retained, coordinates are converted only for physical-distance calculations, and invalid/zero-length road geometry is excluded from candidate search.